In [1]:
import tensorflow as tf
import time
import numpy as np
import os
import pickle5 as pickle
import argparse
import utilityarm as utility
import pandas as pd
from sklearn.metrics import *

In [2]:
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior() 

class BPR:

    def __init__(self, sess, dict_args, train_df, vali_df
                 , key_type, user_type_list, item_type_count):
       
        self.dataname = dict_args['dataname']

        self.key_type = key_type
        self.user_type_list = user_type_list
        self.item_type_count = item_type_count

        self.sess = sess
        #self.args = args

        self.num_cols = len(train_df['item_id'].unique())
        self.num_rows = len(train_df['user_id'].unique())

        self.hidden_neuron = dict_args['hidden_neuron']
        self.neg = dict_args['neg']
        self.batch_size = dict_args['batch_size']

        self.train_df = train_df
        self.vali_df = vali_df
        self.num_train = len(self.train_df)
        self.num_vali = len(self.vali_df)

        self.train_epoch = dict_args['train_epoch']

        self.lr = dict_args['lr'] # learning rate
        self.optimizer_method = dict_args['optimizer_method']
        self.display_step = dict_args['display_step']

        #self.num_genre = dict_args['num_type']

        self.reg = dict_args['reg'] # regularization term trade-off

        print('**********BPR**********')
        #print(self.args)
        self._prepare_model()

    def run(self):
        init = tf.global_variables_initializer()
        self.sess.run(init)
        for epoch_itr in range(1, self.train_epoch + 1):
            self.train_model(epoch_itr)
            if epoch_itr % self.display_step == 0:
                self.test_model(epoch_itr)
        return self.make_records()

    def _prepare_model(self):
        with tf.name_scope("input_data"):
            self.user_input = tf.placeholder(tf.int32, shape=[None, 1], name="user_input")
            self.item_input_pos = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_pos")
            self.item_input_neg = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_neg")

        with tf.variable_scope("BPR", reuse=tf.AUTO_REUSE):
            self.P = tf.get_variable(name="P", initializer=tf.truncated_normal(shape=[self.num_rows, self.hidden_neuron],
                                                                          mean=0, stddev=0.03), dtype=tf.float32)
            self.Q = tf.get_variable(name="Q", initializer=tf.truncated_normal(shape=[self.num_cols+20, self.hidden_neuron],
                                                                          mean=0, stddev=0.03), dtype=tf.float32)

        self.saver = tf.train.Saver([self.P, self.Q])

        p = tf.reduce_sum(tf.nn.embedding_lookup(self.P, self.user_input), 1)
        q_neg = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_neg), 1)
        q_pos = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_pos), 1)

        predict_pos = (tf.reduce_sum(p * q_pos, 1))
        predict_neg = (tf.reduce_sum(p * q_neg, 1))

        cost1 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg)))
        cost2 = self.reg * 0.5 * (self.l2_norm(self.P) + self.l2_norm(self.Q))  # regularization term

        self.cost = cost1 + cost2  # the loss function

        if self.optimizer_method == "Adam":
            optimizer = tf.train.AdamOptimizer(self.lr)
        elif self.optimizer_method == "Adadelta":
            optimizer = tf.train.AdadeltaOptimizer(self.lr)
        elif self.optimizer_method == "Adagrad":
            optimizer = tf.train.AdadeltaOptimizer(self.lr)
        elif self.optimizer_method == "RMSProp":
            optimizer = tf.train.RMSPropOptimizer(self.lr)
        elif self.optimizer_method == "GradientDescent":
            optimizer = tf.train.GradientDescentOptimizer(self.lr)
        elif self.optimizer_method == "Momentum":
            optimizer = tf.train.MomentumOptimizer(self.lr, 0.9)
        else:
            raise ValueError("Optimizer Key ERROR")

        with tf.variable_scope("Optimizer", reuse=tf.AUTO_REUSE):
            self.optimizer = optimizer.minimize(self.cost)

    def train_model(self, itr):
        NS_start_time = time.time() * 1000.0
        epoch_cost = 0
        num_sample, user_list, item_pos_list, item_neg_list = utility.negative_sample(self.train_df, self.num_rows,
                                                                                      self.num_cols, self.neg)
        NS_end_time = time.time() * 1000.0

        start_time = time.time() * 1000.0
        num_batch = int(len(user_list) / float(self.batch_size)) + 1
        random_idx = np.random.permutation(len(user_list))
        for i in range(num_batch):

            # get the indices of the current batch
            if i == num_batch - 1:
                batch_idx = random_idx[i * self.batch_size:]
            elif i < num_batch - 1:
                batch_idx = random_idx[(i * self.batch_size):((i + 1) * self.batch_size)]
            _, tmp_cost = self.sess.run(  # do the optimization by the minibatch
                [self.optimizer, self.cost],
                feed_dict={self.user_input: user_list[batch_idx, :],
                           self.item_input_pos: item_pos_list[batch_idx, :],
                           self.item_input_neg: item_neg_list[batch_idx, :]})
            epoch_cost += tmp_cost

        if itr % self.display_step == 0:
            print ("Training //", "Epoch %d //" % itr, " Total cost = {:.5f}".format(epoch_cost),
                   "Training time : %d ms" % (time.time() * 1000.0 - start_time),
                   "negative Sampling time : %d ms" % (NS_end_time - NS_start_time),
                   "negative samples : %d" % (num_sample))

        ckpt_save_path = "./"+self.dataname+"/BPR_check_points"
        if not os.path.exists(ckpt_save_path):
            os.makedirs(ckpt_save_path)
        self.saver.save(sess, ckpt_save_path + "/check_point.ckpt", global_step=itr)

    def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
        if itr % self.display_step == 0:
            start_time = time.time() * 1000.0
            P, Q = self.sess.run([self.P, self.Q])
            Rec = np.matmul(P, Q.T)

            [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
#             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_type, self.user_type_list,
#                                      self.item_type_count)
            utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
            auc_global = utility.auc_per_user(Rec, self.vali_df, self.train_df)

            print("AUC global is: ", auc_global)
            
#             for k in self.key_type:
#                 print("AUC per %d is:\t[%.7f] "%(k, auc[k]))
#             print("AUC global is: ", auc_global)
            print (
                "Testing //", "Epoch %d //" % itr,
                "Testing time : %d ms" % (time.time() * 1000.0 - start_time))
            print("=" * 200)
        

            filename = './unfairBPR_results/epoch'+ str(itr) +'_Rec_' + self.dataname + '_BPR.npy'
            os.makedirs(os.path.dirname(filename), exist_ok=True)           
            with open(filename, "wb") as f:
                np.save(f, Rec)



    def make_records(self):  # record all the results' details into files
        P, Q = self.sess.run([self.P, self.Q])
        Rec = np.matmul(P, Q.T)

        [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
        return precision, recall, f_score, NDCG, Rec

    @staticmethod
    def l2_norm(tensor):
        return tf.reduce_sum(tf.square(tensor))


Instructions for updating:
non-resource variables are not supported in the long term


In [3]:

#optimizer_method = ['Adam', 'Adadelta', 'Adagrad', 'RMSProp', 'GradientDescent','Momentum'], default='Adam')


train_epoch = 20
display_step = 1
lr = 0.01
reg = 0.1
optimizer_method = 'Adam'
hidden_neuron = 20
n = 1
neg = 5
batch_size = 256
dataname = 'amazon'

In [4]:
dict_args =  {"train_epoch": train_epoch,
            "display_step":display_step,
            "lr":lr,
            "reg":reg,
            "optimizer_method":optimizer_method,
            "hidden_neuron":hidden_neuron,
            "n":n,
            "neg":neg,
            "batch_size":batch_size,
            "dataname":dataname}
dict_args

{'train_epoch': 20,
 'display_step': 1,
 'lr': 0.01,
 'reg': 0.1,
 'optimizer_method': 'Adam',
 'hidden_neuron': 20,
 'n': 1,
 'neg': 5,
 'batch_size': 256,
 'dataname': 'amazon'}

In [5]:
with open('./training_df_amazon.pkl', 'rb') as f:
    train_df = pickle.load(f,encoding='latin1')

# with open('./' + dataname + '/valiing_df.pkl', 'rb') as f:
#     vali_df = pickle.load(f,encoding='latin1')  # for validation
    
with open('./testing_df_amazon.pkl', 'rb') as f:
    test_df = pickle.load(f,encoding='latin1')  # for validation
# vali_df = pickle.load(open('./' + dataname + '/testing_df.pkl'))  # for testing

with open('./key_type_amazon.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')
    
with open('./user_idd_type_list_amazon.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')
    
with open('./type_user_vector_amazon.pkl', 'rb') as f:
    type_user_vector = pickle.load(f,encoding='latin1')

with open('./type_count_amazon.pkl', 'rb') as f:
    type_count = pickle.load(f,encoding='latin1')
    
with open('./item_type_count_amazon.pkl', 'rb') as f:
    item_type_count = pickle.load(f,encoding='latin1')

In [6]:
print(type_count)

[('Male', 5199), ('Female', 2446)]


In [7]:
train_df['item_id'].unique()

array([   0,    1,    2, ..., 2287,   29, 1177], dtype=int64)

In [8]:
train_df.head(20)

,user_id,item_id,rating
0,0,0,1.0
1,1,1,5.0
2,3,1,5.0
3,3,2,4.0
4,1,4,5.0
5,5,1,5.0
6,6,3,5.0
7,8,4,5.0
8,9,4,1.0
9,10,6,5.0


In [9]:
train_df.shape

(18968, 3)

In [10]:
test_df.head(20)

,user_id,item_id,rating
0,2,1,5.0
1,4,3,5.0
2,7,5,5.0
3,12,7,2.0
4,14,4,1.0
5,17,8,4.0
6,25,8,3.0
7,29,8,5.0
8,30,8,4.0
9,31,13,5.0


In [11]:
test_df.shape

(7819, 3)

In [12]:
print(len(user_idd_type_list))

7645


In [13]:
user_idd_type_list

[['Female'],
 ['Male'],
 ['Female'],
 ['Female'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Male'],
 ['Female'],
 ['

In [14]:
print(type_user_vector['Female'].shape)

(1, 7645)


In [15]:
type_user_vector

{'Female': array([[1., 0., 1., ..., 0., 1., 0.]]),
 'Male': array([[0., 1., 0., ..., 1., 0., 1.]])}

In [16]:
len(item_type_count)

2289

In [17]:
item_type_count

[{'Female': 2445, 'Male': 5198},
 {'Female': 2444, 'Male': 5193},
 {'Female': 2444, 'Male': 5195},
 {'Female': 2445, 'Male': 5198},
 {'Female': 2446, 'Male': 5193},
 {'Female': 2439, 'Male': 5191},
 {'Female': 2445, 'Male': 5197},
 {'Female': 2444, 'Male': 5197},
 {'Female': 2446, 'Male': 5196},
 {'Female': 2445, 'Male': 5197},
 {'Female': 2446, 'Male': 5195},
 {'Female': 2446, 'Male': 5194},
 {'Female': 2445, 'Male': 5196},
 {'Female': 2445, 'Male': 5186},
 {'Female': 2444, 'Male': 5196},
 {'Female': 2444, 'Male': 5192},
 {'Female': 2434, 'Male': 5157},
 {'Female': 2442, 'Male': 5192},
 {'Female': 2446, 'Male': 5197},
 {'Female': 2446, 'Male': 5197},
 {'Female': 2446, 'Male': 5195},
 {'Female': 2445, 'Male': 5197},
 {'Female': 2445, 'Male': 5191},
 {'Female': 2443, 'Male': 5199},
 {'Female': 2446, 'Male': 5195},
 {'Female': 2443, 'Male': 5190},
 {'Female': 2445, 'Male': 5191},
 {'Female': 2445, 'Male': 5198},
 {'Female': 2426, 'Male': 5143},
 {'Female': 2446, 'Male': 5198},
 {'Female'

In [18]:
print(type_count)

[('Male', 5199), ('Female', 2446)]


In [19]:
num_item = len(train_df['item_id'].unique())
num_user = len(train_df['user_id'].unique())
num_type = len(key_type)
print('items number : ',num_item)
print('users number : ',num_user)
print('user types : ',key_type)


items number :  2271
users number :  7645
user types :  ['Female', 'Male']


In [20]:
dict_args["num_type"] = len(key_type)

In [21]:
user_type_list = [] #preprocessing to be sure that user types are really the right ones armielle 
for u in range(num_user):
    gl = user_idd_type_list[u]
    tmp = []
    for g in gl:
        if g in key_type:
            tmp.append(g)
    user_type_list.append(tmp)

print(len(user_type_list))

7645


In [22]:
len(user_type_list)

7645

In [23]:
print('*' * 50)
print('number of positive feedback: ' + str(len(train_df)))
print('estimated number of training samples: ' + str(neg * len(train_df)))
print('*' * 50)

**************************************************
number of positive feedback: 18968
estimated number of training samples: 94840
**************************************************


In [24]:
# generate user_type matrix
type_user_indicator = np.zeros((num_type, num_user))

for k in range(num_type):
    type_user_indicator[k,:] = type_user_vector[key_type[k]]

In [25]:
precision = np.zeros(4)
recall = np.zeros(4)
f1 = np.zeros(4)
ndcg = np.zeros(4)
RSP = np.zeros(4)
REO = np.zeros(4)

precision 

array([0., 0., 0., 0.])

In [26]:
len(user_type_list)

7645

In [27]:
train_df['item_id'].unique()

array([   0,    1,    2, ..., 2287,   29, 1177], dtype=int64)

In [28]:
#tf.compat.v1.disable_eager_execution()

for i in range(n):
    #with tf.compat.v1.Session() as sess:
    with tf.Session() as sess:
        bpr = BPR(sess, dict_args, train_df, test_df, key_type, user_type_list, item_type_count)
        [prec_one, rec_one, f_one, ndcg_one, Rec] = bpr.run()


**********BPR**********
Training // Epoch 1 //  Total cost = 65987.17876 Training time : 3001 ms negative Sampling time : 12832 ms negative samples : 94840
precision_1	[0.0015697],	||	 precision_5	[0.0013865],	||	 precision_10	[0.0012034],	||	 precision_15	[0.0011162]
recall_1   	[0.0015697],	||	 recall_5   	[0.0068454],	||	 recall_10   	[0.0118814],	||	 recall_15   	[0.0165250]
f_measure_1	[0.0015697],	||	 f_measure_5	[0.0023060],	||	 f_measure_10	[0.0021854],	||	 f_measure_15	[0.0020911]
ndcg_1     	[0.0015697],	||	 ndcg_5     	[0.0042381],	||	 ndcg_10     	[0.0058536],	||	 ndcg_15     	[0.0070846]
Metrics for user type	 Female
precision_1	[0.0016353],	||	 precision_5	[0.0011447],	||	 precision_10	[0.0011447],	||	 precision_15	[0.0008994]
recall_1	[0.0016353],	||	 recall_5	[0.0057236],	||	 recall_10	[0.0114473],	||	 recall_15	[0.0134914]
ndcg_1	[0.0016353],	||	 ndcg_5	[0.0033915],	||	 ndcg_10	[0.0052582],	||	 ndcg_15	[0.0058052]
AUC per user type	[0.4953815]
Metrics for user type	 Ma

precision_1	[0.0013080],	||	 precision_5	[0.0014127],	||	 precision_10	[0.0011903],	||	 precision_15	[0.0011598]
recall_1   	[0.0012034],	||	 recall_5   	[0.0068542],	||	 recall_10   	[0.0116939],	||	 recall_15   	[0.0171223]
f_measure_1	[0.0012535],	||	 f_measure_5	[0.0023426],	||	 f_measure_10	[0.0021607],	||	 f_measure_15	[0.0021724]
ndcg_1     	[0.0013080],	||	 ndcg_5     	[0.0039803],	||	 ndcg_10     	[0.0055213],	||	 ndcg_15     	[0.0069493]
Metrics for user type	 Female
precision_1	[0.0012265],	||	 precision_5	[0.0017989],	||	 precision_10	[0.0013900],	||	 precision_15	[0.0012265]
recall_1	[0.0008994],	||	 recall_5	[0.0083401],	||	 recall_10	[0.0132461],	||	 recall_15	[0.0177433]
ndcg_1	[0.0012265],	||	 ndcg_5	[0.0046977],	||	 ndcg_10	[0.0062123],	||	 ndcg_15	[0.0073886]
AUC per user type	[0.5105137]
Metrics for user type	 Male
precision_1	[0.0013464],	||	 precision_5	[0.0012310],	||	 precision_10	[0.0010964],	||	 precision_15	[0.0011284]
recall_1	[0.0013464],	||	 recall_5	[0.00

Metrics for user type	 Female
precision_1	[0.0040883],	||	 precision_5	[0.0017989],	||	 precision_10	[0.0014718],	||	 precision_15	[0.0013355]
recall_1	[0.0038839],	||	 recall_5	[0.0087899],	||	 recall_10	[0.0145135],	||	 recall_15	[0.0198283]
ndcg_1	[0.0040883],	||	 ndcg_5	[0.0064660],	||	 ndcg_10	[0.0083261],	||	 ndcg_15	[0.0097289]
AUC per user type	[0.5031700]
Metrics for user type	 Male
precision_1	[0.0061550],	||	 precision_5	[0.0023466],	||	 precision_10	[0.0019619],	||	 precision_15	[0.0016285]
recall_1	[0.0061550],	||	 recall_5	[0.0117330],	||	 recall_10	[0.0196192],	||	 recall_15	[0.0243316]
ndcg_1	[0.0061550],	||	 ndcg_5	[0.0090860],	||	 ndcg_10	[0.0116002],	||	 ndcg_15	[0.0128410]
AUC per user type	[0.5032750]
AUC global is:  0.5032413917838381
Testing // Epoch 11 // Testing time : 59292 ms
Training // Epoch 12 //  Total cost = 66015.27704 Training time : 532 ms negative Sampling time : 10739 ms negative samples : 94840
precision_1	[0.0011772],	||	 precision_5	[0.0008895],	

AUC global is:  0.4917785170769817
Testing // Epoch 16 // Testing time : 62035 ms
Training // Epoch 17 //  Total cost = 65971.92554 Training time : 738 ms negative Sampling time : 18013 ms negative samples : 94840
precision_1	[0.0044474],	||	 precision_5	[0.0018313],	||	 precision_10	[0.0012688],	||	 precision_15	[0.0011424]
recall_1   	[0.0043427],	||	 recall_5   	[0.0089427],	||	 recall_10   	[0.0124744],	||	 recall_15   	[0.0169217]
f_measure_1	[0.0043944],	||	 f_measure_5	[0.0030400],	||	 f_measure_10	[0.0023033],	||	 f_measure_15	[0.0021402]
ndcg_1     	[0.0044474],	||	 ndcg_5     	[0.0066683],	||	 ndcg_10     	[0.0078040],	||	 ndcg_15     	[0.0089757]
Metrics for user type	 Female
precision_1	[0.0024530],	||	 precision_5	[0.0012265],	||	 precision_10	[0.0008585],	||	 precision_15	[0.0007904]
recall_1	[0.0024530],	||	 recall_5	[0.0061325],	||	 recall_10	[0.0085854],	||	 recall_15	[0.0118561]
ndcg_1	[0.0024530],	||	 ndcg_5	[0.0042717],	||	 ndcg_10	[0.0050420],	||	 ndcg_15	[0.005902

In [31]:
list_recom_unfairbpr = []
list_recom_constiemfairadvbpr = []
list_recom_constiemfairadvbpr_bis = []
list_recom_constiemfairadvbpr_01 = []
list_recom_constiemfairadvbpr_02 = []
list_recom_constiemfairadvbpr_06 = []
list_recom_constiemfairadvbpr_07 = []
list_recom_constiemfairadvbpr_08 = []
list_recom_constiemfairadvbpr_09 = []
list_recom_constiemfairadvbpr_001 = []
list_recom_constiemfairadvbpr_0001 = []
list_recom_constiemfairadvbpr_00001 = []
list_recom_constiemfairadvbpr_000001 = []
list_recom_fairadvbpr = []
list_recom_iemfairadvbpr = []
list_recom_uemfairbpr = []

for itr in range(1, 20 + 1):
    filename = './unfairBPR_results/epoch'+ str(itr) +'_Rec_' + dataname + '_BPR.npy'
    with open(filename, "rb") as f:
        recom = np.load(f)
        list_recom_unfairbpr.append(recom)



for itr in range(1, 9 + 1):    
    filename2 = './fairAdvBPR_results/epoch'+ str(itr) +'_Rec_' + dataname + '_fairAdvBPR.npy'
    with open(filename2, "rb") as f:
        recom = np.load(f)
        list_recom_fairadvbpr.append(recom)
"""        
    filename3 = './IembfairAdvBPR_results/epoch'+ str(itr) +'_Rec_' + dataname + '_iembfairAdvBPR.npy'
    with open(filename3, "rb") as f:
        recom = np.load(f)
        list_recom_iemfairadvbpr.append(recom)
        
    filename4 = './UembfairAdvBPR_results/epoch'+ str(itr) +'_Rec_' + dataname + '_uembfairAdvBPR.npy'
    with open(filename4, "rb") as f:
        recom = np.load(f)
        list_recom_uemfairbpr.append(recom)

    filename5 = './const_IembfairAdvBPR_results_bis/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename5, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_bis.append(recom)

    filename6 = './const_IembfairAdvBPR_results_01/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename6, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_01.append(recom)
        
    filename7 = './const_IembfairAdvBPR_results_02/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename7, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_02.append(recom)
        
    filename8 = './const_IembfairAdvBPR_results_001/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename8, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_001.append(recom)
        
    filename9 = './const_IembfairAdvBPR_results_0001/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename9, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_0001.append(recom)
        
    filename10 = './const_IembfairAdvBPR_results_00001/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename10, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_00001.append(recom)
        
    filename11 = './const_IembfairAdvBPR_results_000001/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename11, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_000001.append(recom)
    

for itr in range(32, 51 + 1):
    filename1 = './const_IembfairAdvBPR_adv50_results_05/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename1, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr.append(recom) 
        
    filename2 = './const_IembfairAdvBPR_adv50_results_06/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename2, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_06.append(recom)    
        
    filename3 = './const_IembfairAdvBPR_adv50_results_07/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename3, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_07.append(recom)
        
    filename4 = './const_IembfairAdvBPR_adv50_results_08/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename4, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_08.append(recom)  
        
    filename5 = './const_IembfairAdvBPR_adv50_results_09/epoch'+ str(itr) +'_Rec_' + dataname + '_constfairAdvBPR.npy'
    with open(filename5, "rb") as f:
        recom = np.load(f)
        list_recom_constiemfairadvbpr_09.append(recom)    


"""

'        \n    filename3 = \'./IembfairAdvBPR_results/epoch\'+ str(itr) +\'_Rec_\' + dataname + \'_iembfairAdvBPR.npy\'\n    with open(filename3, "rb") as f:\n        recom = np.load(f)\n        list_recom_iemfairadvbpr.append(recom)\n        \n    filename4 = \'./UembfairAdvBPR_results/epoch\'+ str(itr) +\'_Rec_\' + dataname + \'_uembfairAdvBPR.npy\'\n    with open(filename4, "rb") as f:\n        recom = np.load(f)\n        list_recom_uemfairbpr.append(recom)\n\n    filename5 = \'./const_IembfairAdvBPR_results_bis/epoch\'+ str(itr) +\'_Rec_\' + dataname + \'_constfairAdvBPR.npy\'\n    with open(filename5, "rb") as f:\n        recom = np.load(f)\n        list_recom_constiemfairadvbpr_bis.append(recom)\n\n    filename6 = \'./const_IembfairAdvBPR_results_01/epoch\'+ str(itr) +\'_Rec_\' + dataname + \'_constfairAdvBPR.npy\'\n    with open(filename6, "rb") as f:\n        recom = np.load(f)\n        list_recom_constiemfairadvbpr_01.append(recom)\n        \n    filename7 = \'./const_Iembfair

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from colour import Color

def savepdf_barplot_color_gradient2(ymin = 0.5, ymax = 0.7, whis = 5, start_color='pink',end_color='blue',num_color=5, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    fig = plt.figure()
    gs = fig.add_gridspec(1, 2, hspace=0, wspace=0)
    (ax1, ax2) = gs.subplots(sharex='col', sharey='row')
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    ax1.bar(X_axis, axis_y1, color=colors)
    ax1.hlines(y=axis_y1[0], xmin = 0, xmax = len(axis_x)-1, colors='black', linestyles='--', lw=1)
    
    plt.sca(ax1)
    plt.xticks(X_axis, axis_x, rotation =50)
    #plt.xlabel(xlabel)
    #fig.suptitle(title)
    plt.ylabel(ylabel, fontsize=18)
    plt.rcParams.update({'font.size': 13}) 
    plt.grid()
    
    plt.sca(ax2)
    ax2.boxplot(axis_y1, whis = whis)
    ax1.set_ylim(ymin, ymax)
    
    plt.tight_layout()
    plt.savefig(plot_file)

def savepdf_barplot_color_gradient(ymin = 0.5, ymax = 0.7, start_color='orange',end_color='yellow',num_color=3, title='',axis_x = None, xlabel = '', axis_y1 = None, axis_y2 = None, axis_y3 = None, ylabel ='',barwidth = 0.3, plot_file = '',legendy1 = '',legendy2 = '',legendy3 = ''):
    
    fig, ax1 = plt.subplots()
#     gs = fig.add_gridspec(1, 2, hspace=0, wspace=0)
#     (ax1, ax2) = gs.subplots(sharex='col', sharey='row')
    #fig.suptitle('Sharing x per column, y per row')
    
#     ax1.plot(x + 1, -y, 'tab:green')
#     ax2.plot(x + 2, -y**2, 'tab:red')

#     for ax in fig.get_axes():
#         ax.label_outer()
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    #ax1.bar(X_axis, axis_y1, color=colors)
    ax1.bar(2*X_axis - barwidth, axis_y1, width =barwidth, color=colors[num_color-num_color], label = legendy1)
    ax1.bar(2*X_axis + 0, axis_y2, width = barwidth, color=colors[num_color-2],label = legendy2)
    ax1.bar(2*X_axis + barwidth, axis_y3, width = barwidth, color=colors[num_color-1],label = legendy3)
    
    ax1.set_ylim(ymin, ymax)

    #plt.xlabel(xlabel, fontsize=22)
    plt.ylabel(ylabel, fontsize=18)
    #fig.suptitle(title)
    plt.grid()
    plt.legend(fontsize=12) 
    plt.rcParams.update({'font.size': 12})   
    #ax2 = ax1.twinx()
    #ax1.boxplot([[axis_y1[0],axis_y2[0],axis_y3[0]],[axis_y1[1],axis_y2[1],axis_y3[1]],[axis_y1[2],axis_y2[2],axis_y3[2]]], positions=[2*X_axis[0], 2*X_axis[1] , 2*X_axis[2] ])
    ax1.boxplot([[axis_y1[0],axis_y2[0],axis_y3[0]],[axis_y1[1],axis_y2[1],axis_y3[1]],[axis_y1[2],axis_y2[2],axis_y3[2]],[axis_y1[3],axis_y2[3],axis_y3[3]],[axis_y1[4],axis_y2[4],axis_y3[4]]], positions=[2*X_axis[0], 2*X_axis[1] , 2*X_axis[2], 2*X_axis[3], 2*X_axis[4]])
   
    #ax1.set_xticklabels(axis_x)
    plt.xticks(2*X_axis, axis_x)
    
    plt.tight_layout()
    plt.savefig(plot_file)
    
    
    
    
def savepdf_barplot_color_gradient3(ymin = 0.5, ymax = 0.7, title='',axis_x = None, xlabel = '', axis_y = None, ylabel ='',barwidth = 0.3, plot_file = ''):
    
    fig, ax1 = plt.subplots()
    
    X_axis = np.arange(len(axis_x))
    #ax1.bar(X_axis, axis_y1, color=colors)
    #ax1.bar(2*X_axis - barwidth, axis_y1, width =barwidth, color=colors[num_color-num_color], label = legendy1)
    ax1.bar(2*X_axis + 0, axis_y, width = barwidth, color='orange')
    #ax1.bar(2*X_axis + barwidth, axis_y3, width = barwidth, color=colors[num_color-1],label = legendy3)
    
    ax1.set_ylim(ymin, ymax)

    plt.ylabel(ylabel, fontsize=18)
    #fig.suptitle(title)
    plt.grid()
    plt.legend(fontsize=12) 
    plt.rcParams.update({'font.size': 12})   
    plt.xticks(2*X_axis, axis_x)
    
    plt.tight_layout()
    plt.savefig(plot_file)
    
def plotline_save_as_pdf(title='',axis_x = None, xlabel = '', axis_y1 = None, axis_y2 = None , axis_y3 = None, axis_y4 = None,axis_y5 = None, ylabel ='', linewidth=3, plot_file = '',legendy1='',legendy2='',legendy3='',legendy4='',legendy5=''):
    if axis_y1 is not None:
        plt.plot(axis_x, axis_y1, color='orange', linewidth = linewidth, label = legendy1)
    if axis_y2 is not None:
        plt.plot(axis_x, axis_y2, 'g', linewidth = linewidth, label = legendy2)
    if axis_y3 is not None:
        plt.plot(axis_x, axis_y3, 'b', linewidth = linewidth, label = legendy3)
    if axis_y4 is not None:
        plt.plot(axis_x, axis_y4, 'black', linewidth = linewidth, label = legendy4)
    if axis_y5 is not None:
        plt.plot(axis_x, axis_y5, 'red', linewidth = linewidth, label = legendy5)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid()

    plt.savefig(plot_file)

def plotline_save_as_pdf2(title='',axis_x1 = None,axis_x2 = None, xlabel = '', axis_y1 = None, axis_y2 = None , axis_y3 = None, axis_y4 = None,axis_y5 = None, ylabel ='', linewidth=3, plot_file = '',legendy1='',legendy2='',legendy3='',legendy4='',legendy5=''):
    if axis_y1 is not None:
        plt.plot(axis_x1, axis_y1, color='tab:orange', linewidth = linewidth, label = legendy1)
    if axis_y2 is not None:
        plt.plot(axis_x2, axis_y2, color='tab:green', linewidth = linewidth, label = legendy2)
    if axis_y3 is not None:
        plt.plot(axis_x2, axis_y3, color='tab:blue', linewidth = linewidth, label = legendy3)
    if axis_y4 is not None:
        plt.plot(axis_x2, axis_y4, color='tab:purple', linewidth = linewidth, label = legendy4)
    if axis_y5 is not None:
        plt.plot(axis_x2, axis_y5, color='tab:red', linewidth = linewidth, label = legendy5)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid()

    plt.savefig(plot_file)




In [ ]:
"""
auc_constiemfairadvbpr_01 = []
auc_constiemfairadvbpr_02 = []
auc_constiemfairadvbpr_06 = []
auc_constiemfairadvbpr_07 = []
auc_constiemfairadvbpr_08 = []
auc_constiemfairadvbpr_09 = []
auc_constiemfairadvbpr_001 = []
auc_constiemfairadvbpr_0001 = []
auc_constiemfairadvbpr_00001 = []
auc_constiemfairadvbpr_000001 = []

auc_per_type_constiemfairadvbpr_01 = []
auc_per_type_constiemfairadvbpr_02 = []
auc_per_type_constiemfairadvbpr_06 = []
auc_per_type_constiemfairadvbpr_07 = []
auc_per_type_constiemfairadvbpr_08 = []
auc_per_type_constiemfairadvbpr_09 = []
auc_per_type_constiemfairadvbpr_001 = []
auc_per_type_constiemfairadvbpr_0001 = []
auc_per_type_constiemfairadvbpr_00001 = []
auc_per_type_constiemfairadvbpr_000001 = []

for rec in list_recom_constiemfairadvbpr_01:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_01.append(auc)

    auc_per_type_constiemfairadvbpr_01.append(auc_per_user_type)
    
for rec in list_recom_constiemfairadvbpr_02:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_02.append(auc)

    auc_per_type_constiemfairadvbpr_02.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_001:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_001.append(auc)

    auc_per_type_constiemfairadvbpr_001.append(auc_per_user_type)
    
for rec in list_recom_constiemfairadvbpr_0001:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_0001.append(auc)

    auc_per_type_constiemfairadvbpr_0001.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_00001:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_00001.append(auc)

    auc_per_type_constiemfairadvbpr_00001.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_000001:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_000001.append(auc)

    auc_per_type_constiemfairadvbpr_000001.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_06:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_06.append(auc)

    auc_per_type_constiemfairadvbpr_06.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_07:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_07.append(auc)

    auc_per_type_constiemfairadvbpr_07.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_08:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_08.append(auc)

    auc_per_type_constiemfairadvbpr_08.append(auc_per_user_type)

for rec in list_recom_constiemfairadvbpr_09:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_constiemfairadvbpr_09.append(auc)

    auc_per_type_constiemfairadvbpr_09.append(auc_per_user_type)

"""


In [32]:
auc_plot_unfairbpr=[]
auc_plot_constiemfairbpr=[]
auc_plot_fairadvbpr=[]
auc_plot_iemfairadvbpr=[]
auc_plot_uemfairadvbpr=[]

precision_plot_unfairbpr=[]
precision_plot_constiemfairbpr=[]
precision_plot_fairadvbpr=[]
precision_plot_iemfairadvbpr=[]
precision_plot_uemfairadvbpr=[]

recall_plot_unfairbpr=[]
recall_plot_constiemfairbpr=[]
recall_plot_fairadvbpr=[]
recall_plot_iemfairadvbpr=[]
recall_plot_uemfairadvbpr=[]

ndcg_plot_unfairbpr=[]
ndcg_plot_constiemfairbpr=[]
ndcg_plot_fairadvbpr=[]
ndcg_plot_iemfairadvbpr=[]
ndcg_plot_uemfairadvbpr=[]

precision_perUserType_plot_unfairbpr=[]
precision_perUserType_plot_constiemfairbpr=[]
precision_perUserType_plot_fairadvbpr=[]
precision_perUserType_plot_iemfairadvbpr=[]
precision_perUserType_plot_uemfairadvbpr=[]

recall_perUserType_plot_unfairbpr=[]
recall_perUserType_plot_constiemfairbpr=[]
recall_perUserType_plot_fairadvbpr=[]
recall_perUserType_plot_iemfairadvbpr=[]
recall_perUserType_plot_uemfairadvbpr=[]

ndcg_perUserType_plot_unfairbpr=[]
ndcg_perUserType_plot_constiemfairbpr=[]
ndcg_perUserType_plot_fairadvbpr=[]
ndcg_perUserType_plot_iemfairadvbpr=[]
ndcg_perUserType_plot_uemfairadvbpr=[]

auc_perUserType_plot_unfairbpr=[]
auc_perUserType_plot_constiemfairbpr=[]
auc_perUserType_plot_fairadvbpr=[]
auc_perUserType_plot_iemfairadvbpr=[]
auc_perUserType_plot_uemfairadvbpr=[]

for rec in list_recom_unfairbpr:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_plot_unfairbpr.append(auc)
    precision_plot_unfairbpr.append(precision)
    recall_plot_unfairbpr.append(recall)
    ndcg_plot_unfairbpr.append(ndcg)
    
    precision_perUserType_plot_unfairbpr.append(precision_per_user_type)
    recall_perUserType_plot_unfairbpr.append(recall_per_user_type)
    ndcg_perUserType_plot_unfairbpr.append(ndcg_per_user_type)
    auc_perUserType_plot_unfairbpr.append(auc_per_user_type)

    """"
for rec in list_recom_constiemfairadvbpr:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_plot_constiemfairbpr.append(auc)
    precision_plot_constiemfairbpr.append(precision)
    recall_plot_constiemfairbpr.append(recall)
    ndcg_plot_constiemfairbpr.append(ndcg)
    
    precision_perUserType_plot_constiemfairbpr.append(precision_per_user_type)
    recall_perUserType_plot_constiemfairbpr.append(recall_per_user_type)
    ndcg_perUserType_plot_constiemfairbpr.append(ndcg_per_user_type)
    auc_perUserType_plot_constiemfairbpr.append(auc_per_user_type)
"""
    
for rec in list_recom_fairadvbpr:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_plot_fairadvbpr.append(auc)
    precision_plot_fairadvbpr.append(precision)
    recall_plot_fairadvbpr.append(recall)
    ndcg_plot_fairadvbpr.append(ndcg)
    
    precision_perUserType_plot_fairadvbpr.append(precision_per_user_type)
    recall_perUserType_plot_fairadvbpr.append(recall_per_user_type)
    ndcg_perUserType_plot_fairadvbpr.append(ndcg_per_user_type)
    auc_perUserType_plot_fairadvbpr.append(auc_per_user_type)
    
""""
for rec in list_recom_iemfairadvbpr:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_plot_iemfairadvbpr.append(auc)
    precision_plot_iemfairadvbpr.append(precision)
    recall_plot_iemfairadvbpr.append(recall)
    ndcg_plot_iemfairadvbpr.append(ndcg)
    
    precision_perUserType_plot_iemfairadvbpr.append(precision_per_user_type)
    recall_perUserType_plot_iemfairadvbpr.append(recall_per_user_type)
    ndcg_perUserType_plot_iemfairadvbpr.append(ndcg_per_user_type)
    auc_perUserType_plot_iemfairadvbpr.append(auc_per_user_type)

"""

# for rec in list_recom_uemfairbpr:
#     [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

#     [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
#     auc = utility.auc_per_user(rec, test_df, train_df)
#     auc_plot_uemfairadvbpr.append(auc)
#     precision_plot_uemfairadvbpr.append(precision)
#     recall_plot_uemfairadvbpr.append(recall)
#     ndcg_plot_uemfairadvbpr.append(ndcg)
    
#     precision_perUserType_plot_uemfairadvbpr.append(precision_per_user_type)
#     recall_perUserType_plot_uemfairadvbpr.append(recall_per_user_type)
#     ndcg_perUserType_plot_uemfairadvbpr.append(ndcg_per_user_type)
#     auc_perUserType_plot_uemfairadvbpr.append(auc_per_user_type)

""""
for rec in list_recom_uemfairbpr:
    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)

    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)
    auc = utility.auc_per_user(rec, test_df, train_df)
    auc_plot_uemfairadvbpr.append(auc)
    precision_plot_uemfairadvbpr.append(precision)
    recall_plot_uemfairadvbpr.append(recall)
    ndcg_plot_uemfairadvbpr.append(ndcg)
    
    precision_perUserType_plot_uemfairadvbpr.append(precision_per_user_type)
    recall_perUserType_plot_uemfairadvbpr.append(recall_per_user_type)
    ndcg_perUserType_plot_uemfairadvbpr.append(ndcg_per_user_type)
    auc_perUserType_plot_uemfairadvbpr.append(auc_per_user_type)
    
"""

precision_1	[0.0015697],	||	 precision_5	[0.0013865],	||	 precision_10	[0.0012034],	||	 precision_15	[0.0011162]
recall_1   	[0.0015697],	||	 recall_5   	[0.0068454],	||	 recall_10   	[0.0118814],	||	 recall_15   	[0.0165250]
f_measure_1	[0.0015697],	||	 f_measure_5	[0.0023060],	||	 f_measure_10	[0.0021854],	||	 f_measure_15	[0.0020911]
ndcg_1     	[0.0015697],	||	 ndcg_5     	[0.0042381],	||	 ndcg_10     	[0.0058536],	||	 ndcg_15     	[0.0070846]
Metrics for user type	 Female
precision_1	[0.0016353],	||	 precision_5	[0.0011447],	||	 precision_10	[0.0011447],	||	 precision_15	[0.0008994]
recall_1	[0.0016353],	||	 recall_5	[0.0057236],	||	 recall_10	[0.0114473],	||	 recall_15	[0.0134914]
ndcg_1	[0.0016353],	||	 ndcg_5	[0.0033915],	||	 ndcg_10	[0.0052582],	||	 ndcg_15	[0.0058052]
AUC per user type	[0.4953815]
Metrics for user type	 Male
precision_1	[0.0015388],	||	 precision_5	[0.0015003],	||	 precision_10	[0.0012310],	||	 precision_15	[0.0012182]
recall_1	[0.0015388],	||	 recall_5	[0.00

precision_1	[0.0049706],	||	 precision_5	[0.0016481],	||	 precision_10	[0.0013996],	||	 precision_15	[0.0012296]
recall_1   	[0.0049706],	||	 recall_5   	[0.0080488],	||	 recall_10   	[0.0137170],	||	 recall_15   	[0.0180118]
f_measure_1	[0.0049706],	||	 f_measure_5	[0.0027360],	||	 f_measure_10	[0.0025400],	||	 f_measure_15	[0.0023020]
ndcg_1     	[0.0049706],	||	 ndcg_5     	[0.0065387],	||	 ndcg_10     	[0.0083586],	||	 ndcg_15     	[0.0095099]
Metrics for user type	 Female
precision_1	[0.0044971],	||	 precision_5	[0.0015536],	||	 precision_10	[0.0013083],	||	 precision_15	[0.0011992]
recall_1	[0.0044971],	||	 recall_5	[0.0074952],	||	 recall_10	[0.0125375],	||	 recall_15	[0.0171709]
ndcg_1	[0.0044971],	||	 ndcg_5	[0.0060599],	||	 ndcg_10	[0.0076850],	||	 ndcg_15	[0.0089325]
AUC per user type	[0.5018829]
Metrics for user type	 Male
precision_1	[0.0051933],	||	 precision_5	[0.0016926],	||	 precision_10	[0.0014426],	||	 precision_15	[0.0012438]
recall_1	[0.0051933],	||	 recall_5	[0.00

precision_1	[0.0037933],	||	 precision_5	[0.0020144],	||	 precision_10	[0.0013865],	||	 precision_15	[0.0012383]
recall_1   	[0.0037933],	||	 recall_5   	[0.0100065],	||	 recall_10   	[0.0137999],	||	 recall_15   	[0.0185088]
f_measure_1	[0.0037933],	||	 f_measure_5	[0.0033537],	||	 f_measure_10	[0.0025199],	||	 f_measure_15	[0.0023213]
ndcg_1     	[0.0037933],	||	 ndcg_5     	[0.0069041],	||	 ndcg_10     	[0.0081005],	||	 ndcg_15     	[0.0093315]
Metrics for user type	 Female
precision_1	[0.0028618],	||	 precision_5	[0.0016353],	||	 precision_10	[0.0011856],	||	 precision_15	[0.0009812]
recall_1	[0.0028618],	||	 recall_5	[0.0081766],	||	 recall_10	[0.0118561],	||	 recall_15	[0.0147179]
ndcg_1	[0.0028618],	||	 ndcg_5	[0.0055202],	||	 ndcg_10	[0.0066583],	||	 ndcg_15	[0.0073956]
AUC per user type	[0.5022372]
Metrics for user type	 Male
precision_1	[0.0042316],	||	 precision_5	[0.0021927],	||	 precision_10	[0.0014811],	||	 precision_15	[0.0013592]
recall_1	[0.0042316],	||	 recall_5	[0.01

precision_1	[0.0000000],	||	 precision_5	[0.0002093],	||	 precision_10	[0.0002616],	||	 precision_15	[0.0002703]
recall_1   	[0.0000000],	||	 recall_5   	[0.0010464],	||	 recall_10   	[0.0026161],	||	 recall_15   	[0.0039677]
f_measure_1	[0.0000000],	||	 f_measure_5	[0.0003488],	||	 f_measure_10	[0.0004757],	||	 f_measure_15	[0.0005062]
ndcg_1     	[0.0000000],	||	 ndcg_5     	[0.0005359],	||	 ndcg_10     	[0.0010237],	||	 ndcg_15     	[0.0013821]
Metrics for user type	 Female
precision_1	[0.0000000],	||	 precision_5	[0.0004088],	||	 precision_10	[0.0004906],	||	 precision_15	[0.0004088]
recall_1	[0.0000000],	||	 recall_5	[0.0020442],	||	 recall_10	[0.0049060],	||	 recall_15	[0.0061325]
ndcg_1	[0.0000000],	||	 ndcg_5	[0.0010545],	||	 ndcg_10	[0.0019483],	||	 ndcg_15	[0.0022739]
AUC per user type	[0.5097215]
Metrics for user type	 Male
precision_1	[0.0000000],	||	 precision_5	[0.0001154],	||	 precision_10	[0.0001539],	||	 precision_15	[0.0002052]
recall_1	[0.0000000],	||	 recall_5	[0.00

precision_1	[0.0000000],	||	 precision_5	[0.0002354],	||	 precision_10	[0.0006148],	||	 precision_15	[0.0006104]
recall_1   	[0.0000000],	||	 recall_5   	[0.0009854],	||	 recall_10   	[0.0055941],	||	 recall_15   	[0.0086026]
f_measure_1	[0.0000000],	||	 f_measure_5	[0.0003801],	||	 f_measure_10	[0.0011078],	||	 f_measure_15	[0.0011400]
ndcg_1     	[0.0000000],	||	 ndcg_5     	[0.0005231],	||	 ndcg_10     	[0.0019823],	||	 ndcg_15     	[0.0027811]
Metrics for user type	 Female
precision_1	[0.0000000],	||	 precision_5	[0.0001635],	||	 precision_10	[0.0005315],	||	 precision_15	[0.0004361]
recall_1	[0.0000000],	||	 recall_5	[0.0008177],	||	 recall_10	[0.0049877],	||	 recall_15	[0.0062142]
ndcg_1	[0.0000000],	||	 ndcg_5	[0.0003342],	||	 ndcg_10	[0.0016369],	||	 ndcg_15	[0.0019543]
AUC per user type	[0.5189485]
Metrics for user type	 Male
precision_1	[0.0000000],	||	 precision_5	[0.0002693],	||	 precision_10	[0.0006540],	||	 precision_15	[0.0006924]
recall_1	[0.0000000],	||	 recall_5	[0.00

'"\nfor rec in list_recom_uemfairbpr:\n    [precision, recall, f_score, ndcg] = utility.test_model_all(rec, test_df, train_df)\n\n    [precision_per_user_type, recall_per_user_type, ndcg_per_user_type, auc_per_user_type] = utility.test_model_per_user_type(rec, test_df, train_df, user_type_list, key_type)\n    auc = utility.auc_per_user(rec, test_df, train_df)\n    auc_plot_uemfairadvbpr.append(auc)\n    precision_plot_uemfairadvbpr.append(precision)\n    recall_plot_uemfairadvbpr.append(recall)\n    ndcg_plot_uemfairadvbpr.append(ndcg)\n    \n    precision_perUserType_plot_uemfairadvbpr.append(precision_per_user_type)\n    recall_perUserType_plot_uemfairadvbpr.append(recall_per_user_type)\n    ndcg_perUserType_plot_uemfairadvbpr.append(ndcg_per_user_type)\n    auc_perUserType_plot_uemfairadvbpr.append(auc_per_user_type)\n    \n'

In [ ]:
len(list_recom_constiemfairadvbpr)

In [ ]:
unfairness_constiemfairbpr_05 = []
unfairness_constiemfairbpr_000001 = []
unfairness_constiemfairbpr_00001 = []
unfairness_constiemfairbpr_0001 = []
unfairness_constiemfairbpr_001 = []
unfairness_constiemfairbpr_01 = []
unfairness_constiemfairbpr_02 = []
unfairness_constiemfairbpr_06 = []
unfairness_constiemfairbpr_07 = []
unfairness_constiemfairbpr_08 = []
unfairness_constiemfairbpr_09 = []


unfairness_constiemfairbpr_05.append((auc_perUserType_plot_constiemfairbpr[-1]['M']**2 - auc_perUserType_plot_constiemfairbpr[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_000001.append((auc_per_type_constiemfairadvbpr_000001[-1]['M']**2 - auc_per_type_constiemfairadvbpr_000001[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_00001.append((auc_per_type_constiemfairadvbpr_00001[-1]['M']**2 - auc_per_type_constiemfairadvbpr_00001[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_0001.append((auc_per_type_constiemfairadvbpr_0001[-1]['M']**2 - auc_per_type_constiemfairadvbpr_0001[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_001.append((auc_per_type_constiemfairadvbpr_001[-1]['M']**2 - auc_per_type_constiemfairadvbpr_001[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_01.append((auc_per_type_constiemfairadvbpr_01[-1]['M']**2 - auc_per_type_constiemfairadvbpr_01[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_02.append((auc_per_type_constiemfairadvbpr_02[-1]['M']**2 - auc_per_type_constiemfairadvbpr_02[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_06.append((auc_per_type_constiemfairadvbpr_06[-1]['M']**2 - auc_per_type_constiemfairadvbpr_06[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_07.append((auc_per_type_constiemfairadvbpr_07[-1]['M']**2 - auc_per_type_constiemfairadvbpr_07[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_08.append((auc_per_type_constiemfairadvbpr_08[-1]['M']**2 - auc_per_type_constiemfairadvbpr_08[-1]['F']**2)**2*100)
unfairness_constiemfairbpr_09.append((auc_per_type_constiemfairadvbpr_09[-1]['M']**2 - auc_per_type_constiemfairadvbpr_09[-1]['F']**2)**2*100)


def plotline_save_as_pdf4(title='',axis_x1 = None, xlabel = '', axis_y1 = None, axis_y2 = None, ylabel1 ='', ylabel2 ='', linewidth=2, plot_file = '',legendy1='', legendy2=''):
    
    fig, ax1 = plt.subplots()

    color = 'tab:red'
    ax1.set_xlabel(xlabel)
    ax1.set_ylabel(ylabel1, color=color)
    if axis_y1 is not None:
        #plt.plot(axis_x1, axis_y1, 'b', linewidth = linewidth, label = legendy1)
        lns1 = ax1.plot(axis_x1, axis_y1, color=color, linewidth = linewidth, label = legendy1)
        ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

    color = 'tab:blue'
    ax2.set_ylabel(ylabel2, color=color)  # we already handled the x-label with ax1
    
    if axis_y2 is not None:
       # plt.plot(axis_x1, axis_y2, 'red', linewidth = linewidth, label = legendy2)
        lns2 = ax2.plot(axis_x1, axis_y2, color=color, linewidth = linewidth, label = legendy2)
        ax2.tick_params(axis='y', labelcolor=color)

    lns = lns1 + lns2
    labs = [l.get_label() for l in lns]
    plt.legend(lns, labs, loc=0)
    plt.title(title)
    plt.grid()

    plt.savefig(plot_file)

plotline_save_as_pdf4(title='Unfairness Constant Impact',axis_x1 = np.arange(0,1.1,0.1), xlabel = 'Unfairness Constant', axis_y1 = [auc_constiemfairadvbpr_000001[-1], auc_constiemfairadvbpr_00001[-1], auc_constiemfairadvbpr_0001[-1], auc_constiemfairadvbpr_001[-1], auc_constiemfairadvbpr_01[-1], auc_constiemfairadvbpr_02[-1], auc_plot_constiemfairbpr[-1],auc_constiemfairadvbpr_06[-1], auc_constiemfairadvbpr_07[-1], auc_constiemfairadvbpr_08[-1], auc_constiemfairadvbpr_09[-1]], axis_y2 = [unfairness_constiemfairbpr_000001[-1], unfairness_constiemfairbpr_00001[-1], unfairness_constiemfairbpr_0001[-1], unfairness_constiemfairbpr_001[-1], unfairness_constiemfairbpr_01[-1], unfairness_constiemfairbpr_02[-1], unfairness_constiemfairbpr_05[-1],unfairness_constiemfairbpr_06[-1], unfairness_constiemfairbpr_07[-1], unfairness_constiemfairbpr_08[-1], unfairness_constiemfairbpr_09[-1]], ylabel1 ='AUC', ylabel2 ='Unfairness', linewidth=2, plot_file = './unfairBPR_plots/unfairness_const_impact_per_type.pdf',legendy1='AUC Trend', legendy2='Unfairness Trend')


In [ ]:
 plotline_save_as_pdf2(title='AUC On Testing',axis_x1 = range(1,40+1), axis_x2 = range(21,40+1), xlabel = 'Epoch', axis_y1 = auc_plot_unfairbpr, axis_y2 = auc_plot_constiemfairbpr , axis_y3 = auc_plot_fairadvbpr,axis_y4 = auc_plot_iemfairadvbpr,axis_y5 = auc_plot_uemfairadvbpr, ylabel ='AUC', linewidth=2, plot_file = './unfairBPR_plots/auc_all_methods.pdf',legendy1='UnfairBPR',legendy2='CFairAdvBPR',legendy3='fairAdvBPR',legendy4='iemfairadv',legendy5='uemfairadv')
 

In [ ]:
plotline_save_as_pdf2(title='Precision On Testing',axis_x1 = range(1,40+1), axis_x2 = range(21,40+1), xlabel = 'Epoch', axis_y1 = np.array(precision_plot_unfairbpr)[:,-1], axis_y2 = np.array(precision_plot_constiemfairbpr)[:,-1] , axis_y3 = np.array(precision_plot_fairadvbpr)[:,-1], axis_y4 = np.array(precision_plot_iemfairadvbpr)[:,-1], axis_y5 = np.array(precision_plot_uemfairadvbpr)[:,-1], ylabel ='Precision', linewidth=2, plot_file = './unfairBPR_plots/precisionat15_all_methods.pdf',legendy1='UnfairBPR',legendy2='CFairAdvBPR',legendy3='fairAdvBPR',legendy4='iemfairadv',legendy5='uemfairadv')
 

In [ ]:
plotline_save_as_pdf2(title='NDCG On Testing',axis_x1 = range(1,40+1), axis_x2 = range(21,40+1), xlabel = 'Epoch', axis_y1 = np.array(ndcg_plot_unfairbpr)[:,-1], axis_y2 = np.array(ndcg_plot_constiemfairbpr)[:,-1] , axis_y3 = np.array(ndcg_plot_fairadvbpr)[:,-1], axis_y4 = np.array(ndcg_plot_iemfairadvbpr)[:,-1], axis_y5 = np.array(ndcg_plot_uemfairadvbpr)[:,-1], ylabel ='NDCG', linewidth=2, plot_file = './unfairBPR_plots/ndcgat15_all_methods.pdf',legendy1='UnfairBPR',legendy2='CFairAdvBPR',legendy3='fairAdvBPR',legendy4='iemfairadv',legendy5='uemfairadv')


In [ ]:
#auc_plot_constiemfairbpr[-1]
#axis_y3 = [auc_perUserType_plot_unfairbpr[-1]['F'],auc_perUserType_plot_constiemfairbpr[-1]['F'],auc_perUserType_plot_fairadvbpr[-1]['F'],auc_perUserType_plot_iemfairadvbpr[-1]['F'],auc_perUserType_plot_uemfairadvbpr[-1]['F']]
#axis_y1 = [auc_plot_unfairbpr[-1],auc_plot_constiemfairbpr[-1],auc_plot_fairadvbpr[-1] ,auc_plot_iemfairadvbpr[-1], auc_plot_uemfairadvbpr[-1]]

In [ ]:

savepdf_barplot_color_gradient(ymin = 0.76, ymax = 0.835, start_color='purple',end_color='orange',num_color=3, title='AUC per User Type on Testing (Movielens)',axis_x = ['UnfairBPR','CFairAdvBPR','fairAdvBPR','iemFairAdv','uemFairAdv'], xlabel = '', axis_y1 = [auc_plot_unfairbpr[-1],auc_plot_constiemfairbpr[-1],auc_plot_fairadvbpr[-2] ,auc_plot_iemfairadvbpr[-1], auc_plot_uemfairadvbpr[-1]], axis_y2 = [auc_perUserType_plot_unfairbpr[-1]['M'],auc_perUserType_plot_constiemfairbpr[-1]['M'],auc_perUserType_plot_fairadvbpr[-2]['M'],auc_perUserType_plot_iemfairadvbpr[-1]['M'],auc_perUserType_plot_uemfairadvbpr[-1]['M']], axis_y3 = [auc_perUserType_plot_unfairbpr[-1]['F'],auc_perUserType_plot_constiemfairbpr[-1]['F'],auc_perUserType_plot_fairadvbpr[-2]['F'],auc_perUserType_plot_iemfairadvbpr[-1]['F'],auc_perUserType_plot_uemfairadvbpr[-1]['F']], ylabel ='AUC',barwidth = 0.3, plot_file = './unfairBPR_plots/auc_05_per_type.pdf',legendy1='Overall',legendy2='Male',legendy3='Female') 

In [ ]:
unfairness_unfairbpr = []
unfairness_constiemfairbpr = []
unfairness_fairadvbpr = []
unfairness_iemfairadvbpr = []
unfairness_uemfairadvbpr = []

for i in range(len(auc_perUserType_plot_unfairbpr)):
    unfairness_unfairbpr.append((auc_perUserType_plot_unfairbpr[i]['M']**2 - auc_perUserType_plot_unfairbpr[i]['F']**2)**2*100)

for i in range(len(auc_perUserType_plot_constiemfairbpr)):
    unfairness_constiemfairbpr.append((auc_perUserType_plot_constiemfairbpr[i]['M']**2 - auc_perUserType_plot_constiemfairbpr[i]['F']**2)**2*100)

for i in range(len(auc_perUserType_plot_fairadvbpr)):
    unfairness_fairadvbpr.append((auc_perUserType_plot_fairadvbpr[i]['M']**2 - auc_perUserType_plot_fairadvbpr[i]['F']**2)**2*100)

for i in range(len(auc_perUserType_plot_iemfairadvbpr)):
    unfairness_iemfairadvbpr.append((auc_perUserType_plot_iemfairadvbpr[i]['M']**2 - auc_perUserType_plot_iemfairadvbpr[i]['F']**2)**2*100)
    
for i in range(len(auc_perUserType_plot_uemfairadvbpr)):
    unfairness_uemfairadvbpr.append((auc_perUserType_plot_uemfairadvbpr[i]['M']**2 - auc_perUserType_plot_uemfairadvbpr[i]['F']**2)**2*100)

In [ ]:
#savepdf_barplot_color_gradient3(ymin = 0.76, ymax = 0.835, title='Unfairness per User Type on Testing',axis_x = ['UnfairBPR','CFairAdvBPR','fairAdvBPR','iemFairAdv','uemFairAdv'], xlabel = '', axis_y = [auc_plot_unfairbpr[-1],auc_plot_constiemfairbpr[-1],auc_plot_fairadvbpr[-2] ,auc_plot_iemfairadvbpr[-1], auc_plot_uemfairadvbpr[-1]], ylabel ='Unfairness',barwidth = 0.3, plot_file = './unfairBPR_plots/unfairness_per_type.pdf')
plotline_save_as_pdf2(title='Unfairness per User Type on Testing (Gender)',axis_x1 = range(1,40+1), axis_x2 = range(21,40+1), xlabel = 'Epoch', axis_y1 = unfairness_unfairbpr, axis_y2 = unfairness_constiemfairbpr , axis_y3 = unfairness_fairadvbpr, axis_y4 = unfairness_iemfairadvbpr, axis_y5 = unfairness_uemfairadvbpr, ylabel ='Unfairness', linewidth=2, plot_file = './unfairBPR_plots/unfairness_per_type.pdf',legendy1='UnfairBPR',legendy2='CFairAdvBPR',legendy3='fairAdvBPR',legendy4='iemfairadv',legendy5='uemfairadv')


In [ ]:
savepdf_barplot_color_gradient(ymin = 0.1, ymax = 0.2, start_color='purple',end_color='orange',num_color=3, title='Precision per User Type on Testing (Movielens)',axis_x = ['UnfairBPR','CFairAdvBPR','fairAdvBPR','iemFairAdv','uemFairAdv'], xlabel = '', axis_y1 = [precision_plot_unfairbpr[-1][2], precision_plot_constiemfairbpr[-10][2],precision_plot_fairadvbpr[-1][2] ,precision_plot_iemfairadvbpr[-1][2], precision_plot_uemfairadvbpr[-1][2]], axis_y2 = [precision_perUserType_plot_unfairbpr[-1]['M'][0].tolist()[2],precision_perUserType_plot_constiemfairbpr[-1]['M'][0].tolist()[2],precision_perUserType_plot_fairadvbpr[-1]['M'][0].tolist()[2],precision_perUserType_plot_iemfairadvbpr[-1]['M'][0].tolist()[2],precision_perUserType_plot_uemfairadvbpr[-1]['M'][0].tolist()[2]], axis_y3 = [precision_perUserType_plot_unfairbpr[-1]['F'][0].tolist()[2], precision_perUserType_plot_constiemfairbpr[-10]['F'][0].tolist()[1],precision_perUserType_plot_fairadvbpr[-20]['F'][0].tolist()[2],precision_perUserType_plot_iemfairadvbpr[-20]['F'][0].tolist()[2],precision_perUserType_plot_uemfairadvbpr[-20]['F'][0].tolist()[2]], ylabel ='Precision',barwidth = 0.3, plot_file = './unfairBPR_plots/precision_05_per_type.pdf',legendy1='Overall',legendy2='Male',legendy3='Female') 

In [ ]:
savepdf_barplot_color_gradient(ymin = 0.1, ymax = 0.25, start_color='purple',end_color='orange',num_color=3, title='NDCG per User Type on Testing (Movielens)',axis_x = ['UnfairBPR','CFairAdvBPR','fairAdvBPR','iemFairAdv','uemFairAdv'], xlabel = '', axis_y1 = [ndcg_plot_unfairbpr[-1][2],ndcg_plot_constiemfairbpr[1][2],ndcg_plot_fairadvbpr[-1][2] ,ndcg_plot_iemfairadvbpr[-1][2], ndcg_plot_uemfairadvbpr[-1][2]], axis_y2 = [ndcg_perUserType_plot_unfairbpr[-1]['M'][0].tolist()[2],ndcg_perUserType_plot_constiemfairbpr[1]['M'][0].tolist()[3],ndcg_perUserType_plot_fairadvbpr[-1]['M'][0].tolist()[2], ndcg_perUserType_plot_iemfairadvbpr[-1]['M'][0].tolist()[2],ndcg_perUserType_plot_uemfairadvbpr[-1]['M'][0].tolist()[2]], axis_y3 = [ndcg_perUserType_plot_unfairbpr[-1]['F'][0].tolist()[3], ndcg_perUserType_plot_constiemfairbpr[-20]['F'][0].tolist()[1],ndcg_perUserType_plot_fairadvbpr[-20]['F'][0].tolist()[2], ndcg_perUserType_plot_iemfairadvbpr[-20]['F'][0].tolist()[2], ndcg_perUserType_plot_uemfairadvbpr[-20]['F'][0].tolist()[2]], ylabel ='NDCG',barwidth = 0.3, plot_file = './unfairBPR_plots/ndcg_05_per_type.pdf',legendy1='Overall',legendy2='Male',legendy3='Female') 

In [ ]:
precision_perUserType_plot_fairadvbpr[-1]